# Phase 2: early answering and linear probes
Use completed local Phase 1 records from the exact small-model revision. API traces and synthetic pipeline fixtures are rejected. Early-answer curves and probes measure answer consistency and decodability; neither alone proves that a decision was irrevocably settled.

## Setup
Select **Runtime → Change runtime type → T4 GPU**. Upload the project ZIP containing a top-level `hint-faithfulness/` folder. You may select a saved results ZIP at the same time; the notebook restores it after the project, so you do not need to repack archives. Include local Phase 1 records when continuing an experiment. Do not include credentials or virtual environments. Free GPU availability is not guaranteed.

In [ ]:
from google.colab import files
from pathlib import Path
import io, os, zipfile

# Select the project ZIP and any saved results ZIP together.
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
if not archives:
    raise ValueError("Upload the project ZIP, optionally with a saved results ZIP")
root = Path("/content").resolve()
def contains_project(name):
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
        return "hint-faithfulness/src/run.py" in archive.namelist()
# Install project files first; restore saved data and human labels afterward.
archives.sort(key=lambda name: 0 if contains_project(name) else 1)
for name in archives:
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
        for member in archive.infolist():
            destination = (root / member.filename).resolve()
            if not destination.is_relative_to(root):
                raise ValueError("Archive path escapes /content")
        archive.extractall(root)
project = root / "hint-faithfulness"
if not (project / "src" / "run.py").exists():
    raise ValueError("Project ZIP must contain hint-faithfulness/src/run.py")
os.chdir(project)


In [ ]:
!python -m pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before loading weights"
print(torch.cuda.get_device_name(0))
!python -m unittest discover -s tests -q


## Predict before running
Write your own prediction below. This is a dated expectation, not a result. Keep it when the outcome disagrees.

In [ ]:
from datetime import datetime, timezone
PREDICTION = ""  # Fill in your prediction before running the experiment.
assert PREDICTION.strip(), "Write a prediction first"
with Path("LOG.md").open("a") as log:
    log.write(f"\n\n{datetime.now(timezone.utc).isoformat()} — prediction\n{PREDICTION}\n")


## Early answering
Five token-preserving cuts close the thinking block and request an answer letter. The output retains full-vocabulary probability mass as well as A–D-normalized probabilities. Hinted curves use only observed switches; the unhinted curve uses the first baseline. Intervals resample questions.

In [ ]:
RECORDS = "data/runs/small.jsonl"
assert Path(RECORDS).exists(), "Include local Phase 1 JSONL in your uploaded ZIP"
!python -m src.phase2 early --records {RECORDS} --out-dir results/phase2 --device cuda


In [ ]:
from IPython.display import display, Image
display(Image("results/phase2/early_answering.png"))


## Probe before reasoning
All variants of a question stay in either training or the held-out 30%. Standardization is fit only on training data; regularization is fixed. The plotted layer numbers identify post-block residual states before final RMSNorm. The summary also compares predicted model answers with correct answers on mistakes and evaluates cases where the hint was resisted. Do not select the best layer using test scores and then report it as independently validated.

In [ ]:
!python -m src.phase2 probe --records {RECORDS} --out-dir results/phase2 --device cuda
display(Image("results/phase2/probe_accuracy.png"))


## Interpret the result
Write one paragraph answering: how much of the final answer was recoverable before reasoning, whether hints changed that pattern, and whether the wrong-answer/resisted-hint controls support your interpretation. Include sample counts, failed generations, and the difference between prediction and causal evidence.

## Save your work
Download results before the Colab session ends. The archive includes data, logs, and experiment outputs, but excludes downloaded weights and credentials.

In [ ]:
from google.colab import files
with zipfile.ZipFile("research-results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in (Path("data"), Path("results")):
        if folder.exists():
            for item in folder.rglob("*"):
                if item.is_file():
                    archive.write(item, Path("hint-faithfulness") / item)
    if Path("LOG.md").exists():
        archive.write("LOG.md", "hint-faithfulness/LOG.md")
files.download("research-results.zip")
